# EU Conventions Tree Traversal Examples

This notebook shows how to use the LATTICE LLM-guided traversal flow on the EU conventions semantic tree.

It is intentionally example-oriented:
- no gold paths
- no evaluation metrics
- no tests
- just load a tree, run example queries, inspect the traversal, and visualize the prediction tree

Before running the traversal cells, make sure the API key for your selected LLM backend is available in the environment.


In [ ]:
from __future__ import annotations

import asyncio
import os
import pickle
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists() and (current / "trees").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from google.genai import types as genai_types

from hyperparams import HyperParams
from llm_apis import GenAIAPI, OpenAIResponsesAPI, VllmAPI, LocalModelAPI
from prompts import get_traversal_prompt_response_constraint
from tree_construction.build_llm_bottom_up_tree import run_coro_sync
from tree_objects import InferSample, SemanticNode
from utils import compute_node_registry, get_all_leaf_nodes_with_path, post_process, setup_logger, visualize_sample

REPO_ROOT


In [ ]:
# Configuration
#
# TREE_PATH prefers the expected EU conventions tree path. The fallback points to
# the tree path produced by the current bottom-up construction notebook in this repo.
default_tree_path = REPO_ROOT / "trees" / "EU" / "eu_conventions_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"
if not default_tree_path.exists():
    default_tree_path = REPO_ROOT / "trees" / "EU" / "codigo_civil_notebook" / "eu_conventions_tree-bottom-up-llm.pkl"

# HyperParams.from_args(...) mirrors the way run.py configures retrieval.
# We pass the required args first, then override/add notebook-specific values.
hp = HyperParams.from_args("--subset fiqa --tree_version eu_conventions_notebook")
hp.TREE_PATH = str(default_tree_path)
hp.DATASET = "EU"
hp.LLM_API_BACKEND = "localModel"  # one of: openai, genai, vllm, localModel
hp.LLM = "Qwen/Qwen2.5-1.5B-Instruct" # "google/gemma-2-2b-it"  #"google/gemma-2-2b-it" , "gpt-4.1"
hp.LLM_API_TIMEOUT = 120
hp.LLM_API_MAX_RETRIES = 4
hp.LLM_MAX_CONCURRENT_CALLS = 16
hp.LLM_API_STAGGERING_DELAY = 0.05
hp.VLLM_BASE_URL = "http://localhost:8000/v1"
hp.REASONING_IN_TRAVERSAL_PROMPT = -1
hp.SUBSET = "fiqa"  # chooses the traversal relevance definition
hp.MAX_BEAM_SIZE = 8
hp.SEARCH_WITH_PATH_RELEVANCE = True
hp.NUM_LEAF_CALIB = 0
hp.RELEVANCE_CHAIN_FACTOR = 0.5
hp.MAX_PROMPT_PROTO_SIZE = None
hp.MAX_DOC_DESC_CHAR_LEN = None

LOG_DIR = REPO_ROOT / "results" / "tree_construction"
LOG_DIR.mkdir(parents=True, exist_ok=True)
logger = setup_logger("eu_conventions_traversal_notebook", str(LOG_DIR / "eu_conventions_traversal_notebook.log"))

hp


In [ ]:
tree_path = Path(hp.TREE_PATH)
tree_obj = pickle.loads(tree_path.read_bytes())
semantic_root_node = SemanticNode().load_dict(tree_obj) if isinstance(tree_obj, dict) else tree_obj
node_registry = compute_node_registry(semantic_root_node)
all_leaf_nodes = get_all_leaf_nodes_with_path(semantic_root_node)

print(f"Loaded tree from: {tree_path}")
print(f"Total nodes in semantic tree: {len(node_registry)}")
print(f"Total leaf nodes: {len(all_leaf_nodes)}")
print("Root description preview:")
print(semantic_root_node.desc[:800])


In [ ]:
def make_llm_api(hp, logger):
    if hp.LLM_API_BACKEND == "genai":
        llm_api = GenAIAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
    elif hp.LLM_API_BACKEND == "openai":
        llm_api = OpenAIResponsesAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, log_api_calls=True)
    elif hp.LLM_API_BACKEND == "vllm":
        llm_api = VllmAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, base_url=hp.VLLM_BASE_URL)
    elif hp.LLM_API_BACKEND == "localModel":
        llm_api = LocalModelAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES, log_api_calls=True)
    else:
        raise ValueError(f"Unknown backend: {hp.LLM_API_BACKEND}")

    llm_api_kwargs = {
        "max_concurrent_calls": hp.LLM_MAX_CONCURRENT_CALLS,
        "response_mime_type": "application/json",
        "response_schema": get_traversal_prompt_response_constraint(bool(hp.REASONING_IN_TRAVERSAL_PROMPT)),
        "staggering_delay": hp.LLM_API_STAGGERING_DELAY,
        "print_summary_report": False,
        "thinking_config": genai_types.ThinkingConfig(thinking_budget=hp.REASONING_IN_TRAVERSAL_PROMPT),
    }

    if hp.LLM_API_BACKEND == "vllm":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")
        llm_api_kwargs.pop("response_schema")

    if hp.LLM_API_BACKEND == "openai":
        llm_api_kwargs.pop("response_mime_type")
        llm_api_kwargs.pop("thinking_config")

    return llm_api, llm_api_kwargs


llm_api, llm_api_kwargs = make_llm_api(hp, logger)
print(type(llm_api).__name__)


## Example Queries

Pick one of these or write your own EU Convention query in the next cell.


In [ ]:
EXAMPLE_QUERIES = [
    "What does Article 6 say about the right to a fair trial?",
    "When can the right to liberty and security be restricted?",
    "What protections does Article 8 give for private and family life?",
    "What does the Convention say about prohibition of torture?",
    "How does Protocol 1 protect property rights?",
    "What are the rules for individual applications to the European Court of Human Rights?",
]

for idx, query in enumerate(EXAMPLE_QUERIES):
    print(f"[{idx}] {query}")


In [ ]:
query = EXAMPLE_QUERIES[0]
num_iters = 12
query


In [ ]:
def make_sample(query: str) -> InferSample:
    return InferSample(
        semantic_root_node,
        node_registry,
        hp=hp,
        logger=logger,
        query=query,
        gold_paths=[],
        excluded_ids_set=set(),
    )


async def run_one_iteration_async(sample: InferSample):
    inputs = sample.get_step_prompts()
    prompts = [prompt for prompt, _ in inputs]
    slates = [slate for _, slate in inputs]

    raw_responses = await llm_api.run_batch(prompts, **llm_api_kwargs)
    response_jsons = [post_process(output, return_json=True) for output in raw_responses]
    sample.update(slates, response_jsons)
    return raw_responses, response_jsons


def run_one_iteration(sample: InferSample):
    return run_coro_sync(run_one_iteration_async(sample))


def print_frontier(sample: InferSample):
    print("Current beam state paths:")
    for state_path in sample.beam_state_paths:
        node = state_path[-1]
        print(f"  path={node.path} | path_relevance={node.path_relevance:.3f} | desc={node.desc[:180]}")


def print_top_predictions(sample: InferSample, k: int = 5):
    top_preds = sample.get_top_predictions(k=k)
    if not top_preds:
        print("No leaf predictions yet. Increase num_iters.")
        return

    for rank, (node, score) in enumerate(top_preds, start=1):
        print(f"[{rank}] path={node.path} | score={score:.3f} | id={node.id}")
        print(node.desc[:600])
        print("-" * 120)


async def run_query_async(query: str, num_iters: int = 4, detailed_logs: bool = False) -> InferSample:
    sample = make_sample(query)
    if detailed_logs: print(f"Running traversal for query: {query}")
    for step in range(num_iters):
        print(f"\n--- Iteration {step + 1} ---")
        _, response_jsons = await run_one_iteration_async(sample)
        if(detailed_logs): print_frontier(sample)
        if response_jsons:
            first_reasoning = response_jsons[0].get("reasoning", "")
            if(detailed_logs):
                print("\nReasoning preview:")
                print(str(first_reasoning)[:800])
    return sample


def run_query(query: str, num_iters: int = 4, detailed_logs: bool = False) -> InferSample:
    return run_coro_sync(run_query_async(query, num_iters=num_iters, detailed_logs=detailed_logs))


In [ ]:
#sample = run_query(query, num_iters=num_iters)


In [ ]:
#print_top_predictions(sample, k=5)


## ECtHR Multi-Case Article Retrieval Test

This section loads ECtHR cases, joins each case's fact paragraphs into one LATTICE query, retrieves Convention/Protocol article leaves from the EU conventions tree, and compares the predicted article set with the dataset's multi-label article targets.

The main count is `all_gold_found`: a case is counted as correct when every expected article label appears somewhere in the top retrieved article set. Because a case may involve multiple articles, the table also reports partial matches, precision, recall, and F1.


In [ ]:
from llm_rl_playground.ecthr_evaluation import (
    EcthrTraversalEvaluator,
    get_label_names,
    load_ecthr_dataset,
    summarize_ecthr_cases,
)

ecthr_evaluator = EcthrTraversalEvaluator(
    semantic_root_node=semantic_root_node,
    node_registry=node_registry,
    hp=hp,
    logger=logger,
    llm_api=llm_api,
    llm_api_kwargs=llm_api_kwargs,
)

print(type(ecthr_evaluator).__name__)


In [ ]:
# ECtHR evaluation helpers now live in `llm_rl_playground.ecthr_evaluation`.
# Reuse `ecthr_evaluator.evaluate_ecthr_cases_batched(...)` from any notebook
# after constructing it with that notebook's tree, hp, logger, and llm_api.


In [ ]:
# Configure the ECtHR evaluation run.
# Use split="train" if you want to test quickly on the training set facts,
# or split="test" for the held-out ECtHR Cases test split.
ECTHR_CONFIG = "alleged-violation-prediction"
ECTHR_SPLIT = "train"
N_CASES = 1
NUM_ITERS_PER_CASE = 9
TOP_K_LEAVES = 10
PREDICTION_MIN_SCORE = 0.4  # Try 0.25 or 0.35 for stricter precision filtering.
MAX_PREDICTED_ARTICLES = None  # Lower values usually improve precision and reduce recall.
USE_LLM_SELECTOR = True  # Adds one final LLM call per case to select from LATTICE candidates.

ecthr_dataset = load_ecthr_dataset(split=ECTHR_SPLIT, config=ECTHR_CONFIG)
ecthr_label_names = get_label_names(ecthr_dataset)

print(ecthr_dataset)
print("Label names:", ecthr_label_names)
print("First example labels:", ecthr_dataset[0]["labels"])

results_df, ecthr_results = ecthr_evaluator.evaluate_ecthr_cases_batched(
    ecthr_dataset,
    ecthr_label_names,
    n_cases=N_CASES,
    num_iters=NUM_ITERS_PER_CASE,
    top_k=TOP_K_LEAVES,
    prediction_min_score=PREDICTION_MIN_SCORE,
    max_predicted_articles=MAX_PREDICTED_ARTICLES,
    use_llm_selector=USE_LLM_SELECTOR,
)

results_df


In [ ]:
summarize_ecthr_cases(results_df)


In [ ]:
ecthr_dataset[0]

In [ ]:
ecthr_results

In [ ]:
# Optional: visualize the prediction tree explored by the LLM.
VIS_DIR = REPO_ROOT / "results" / "tree_construction"
VIS_DIR.mkdir(parents=True, exist_ok=True)
html_path = VIS_DIR / "eu_conventions_traversal_prediction_tree.html"

fig = visualize_sample(sample, width=1400, height=900, save_path=str(html_path))
print(html_path)


## Notes

- `num_iters` controls how deep the traversal goes before you inspect results.
- If `print_top_predictions(...)` says there are no leaf predictions yet, increase `num_iters`.
- `hp.SUBSET` controls the relevance definition used inside the traversal prompt. For legal QA-style usage, `fiqa` is usually a reasonable default here.
- `visualize_sample(...)` shows the explored prediction tree, not the static semantic tree structure.
- The default tree path first checks `trees/EU/eu_conventions_notebook/tree-bottom-up-llm.pkl`, then falls back to the existing `trees/EU/codigo_civil_notebook/tree-bottom-up-llm.pkl` path.

- The ECtHR section uses `AUEB-NLP/ecthr_cases` (`alleged-violation-prediction` by default). The case facts are joined into one query and the label list is treated as the expected multi-article target set.
